---
  title: "Aula 4"
  author: Mauro Alixandrini
  date: 2026-16-09
  categories: [GeoAI, technology]
  image: ambiente.jpg
---

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alixandrini/PPEC083/blob/main/Notebooks/Aula_4.ipynb)

# Mapeamento Interativo e Visualização

A visualização é uma das etapas mais importantes em qualquer fl uxo de trabalho de GeoAI. Antes de treinar um modelo, você precisa inspecionar suas imagens, verificar se os rótulos estão alinhados com as feições no terreno e entender a distribuição espacial dos seus dados. Após a inferência, você precisa avaliar as predições visualmente, compará-las com a verdade de campo e comunicar resultados a partes interessadas que podem não ler código. Ao longo de todo o ciclo de vida do GeoAI, desde a aquisição de dados até a implantação do modelo, a capacidade de criar mapas claros e informativos é o que separa profissionais efi cazes daqueles que lutam para interpretar seus resultados.


Gráficos estáticos funcionam para verificações simples, mas dados geoespaciais exigem interatividade. Você precisa navegar por grandes cenas, dar zoom em bairros específicos, alternar camadas e comparar datasets lado a lado. Uma miniatura estática de uma imagem de satélite pode mostrar padrões gerais, mas não pode revelar a pegada de edifi cação desalinhada no canto nordeste da sua área de estudo ou a sombra de nuvem que seu modelo confunde com água.


A biblioteca  leafmap 60  fornece uma interface Pythônica para construir mapas interativos diretamente em notebooks Jupyter, combinando o poder do ipyleafl et com funções convenientes para carregar dados geoespaciais raster, vetoriais e hospedados na nuvem. Ela foi projetada com fluxos de trabalho de GeoAI em mente, oferecendo integração perfeita com os formatos de dados e ferramentas de análise abordados aqui.

In [12]:
import os
import sys

# Aponta dinamicamente para a pasta proj dentro do seu ambiente conda
conda_prefix = sys.prefix
proj_path = os.path.join(conda_prefix, "Library", "share", "proj")

# Configura as variáveis de ambiente necessárias
os.environ["PROJ_DATA"] = proj_path
os.environ["PROJ_LIB"] = proj_path

# Agora importe o pyproj e o leafmap
import pyproj
import leafmap

print("Caminho do PROJ configurado para:", pyproj.datadir.get_data_dir())

Caminho do PROJ configurado para: C:\Users\mauro\AppData\Roaming\Python\Python314\site-packages\pyproj\proj_dir\share\proj


## Criando Mapas Interativos com Leafmap

### Seu Primeiro Mapa

A maneira mais simples de criar um mapa interativo é chamar  leafmap.Map() . Isso produz um widget de mapa com controles de navegação, zoom e camadas integrados. O mapa resultante é totalmente interativo dentro de um notebook Jupyter: você pode clicar e arrastar para navegar, rolar para dar zoom e usar os controles integrados para alternar entre camadas ou ajustar a visualização. Essa interatividade imediata é o que torna o  leafmap  tão valioso para análise exploratória de dados em fluxos de trabalho de GeoAI. Você pode confi gurar o centro do mapa, nível de zoom e dimensões de exibição ao criar o mapa.

In [13]:
import leafmap
m = leafmap.Map(center=[-12.9086, -38.3225], zoom=14, height="600px")
m

Map(center=[-12.9086, -38.3225], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

### Adicionando Mapas Base

Mapas base fornecem o contexto geográfi co que torna as visualizações geoespaciais signifi cativas. Sem um mapa base, um agrupamento de edifi cações detectadas é apenas uma coleção de polígonos flutuando em espaço vazio. Com um mapa base de satélite por baixo, esses mesmos polígonos se tornam claramente interpretáveis como um bairro, um distrito comercial ou um campus universitário.


O  leafmap   fornece  acesso  a  centenas  de  mapas  base  através  do  método  ```add_basemap()``` .  Opções comuns incluem imagens de satélite, mapas de terreno e camadas temáticas. Escolher o mapa base certo depende da sua tarefa: imagens de satélite funcionam melhor para verificar predições de modelo contra feições no terreno, enquanto mapas topográficos são mais apropriados para entender o contexto de terreno em análises de elevação ou hidrológicas.

In [14]:
m = leafmap.Map(center=[-12.9086, -38.3225], zoom=14, height="600px")
m.add_basemap("Esri.WorldImagery")
m

Map(center=[-12.9086, -38.3225], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

Para ver todos os mapas base disponíveis, inspecione o dicionário  ```leafmap.basemaps``` :

In [15]:
basemaps = list(leafmap.basemaps.keys())
print(f"Total basemaps: {len(basemaps)}")
print("First 10:", basemaps[10:20])

Total basemaps: 155
First 10: ['NLCD 2011 CONUS Land Cover', 'NLCD 2008 CONUS Land Cover', 'NLCD 2006 CONUS Land Cover', 'NLCD 2004 CONUS Land Cover', 'NLCD 2001 CONUS Land Cover', 'USGS NAIP Imagery', 'USGS NAIP Imagery False Color', 'USGS NAIP Imagery NDVI', 'USGS Hydrography', 'USGS 3DEP Elevation']


Você também pode adicionar múltiplos mapas base e alternar entre eles usando o controle de camadas:

In [16]:
m = leafmap.Map()
m.add_basemap("Esri.WorldImagery")
m.add_basemap("OpenTopoMap")
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## Trabalhando com Dados Raster em Mapas

Dados raster formam a espinha dorsal da maioria dos fluxos de trabalho de GeoAI. Imagens de satélite, modelos digitais de elevação, mapas de temperatura da superfície terrestre e saídas de predição de modelos são todos armazenados como rasters. Ser capaz de exibir essas camadas em um mapa interativo é essencial para controle de qualidade, análise exploratória e interpretação de resultados. O  leafmap  fornece vários métodos para adicionar dados raster a mapas, cada um otimizado para diferentes fontes de dados e padrões de acesso.

### Adicionando Camadas GeoTIFF

O método  ```add_raster()```  carrega um GeoTIFF local ou remoto e o exibe no mapa com um colormap de sua escolha. Este método lê os dados raster, reprojeta se necessário e renderiza como uma camada de tiles no mapa. Para rasters de banda única como modelos de altura ou saídas de classifi cação, você pode aplicar qualquer colormap do matplotlib para mapear valores de pixel a cores.


Ao longo deste capítulo, usamos quatro datasets de um projeto de detecção de edifi cações em Las Vegas. Primeiro os baixamos localmente usando a biblioteca  geoai , depois os visualizamos com  leafmap . Os quatro datasets são:

* Imagem aérea NAIP (GeoTIFF de 4 bandas): bandas Vermelho, Verde, Azul e Infravermelho Próximo a 60 cm de resolução
* Altura Acima do Solo (HAG) (GeoTIFF de banda única): altura da superfície derivada de LiDAR em metros
* Pegadas de edifi cações  (GeoJSON): polígonos vetoriais delineando edifi cações individuais
* Máscara de edifi cações  (GeoTIFF de banda única): uma máscara binária rasterizada onde 1 indica pixels de edificação e 0 indica fundo

In [17]:
import geoai
import leafmap

In [18]:
import os
import sys

# Verifica se o arquivo proj.db existe dentro do ambiente conda
expected_db = os.path.join(sys.prefix, "Library", "share", "proj", "proj.db")
print("O arquivo proj.db existe?", os.path.exists(expected_db))
print("Caminho esperado:", expected_db)

# Tenta importar direto o componente que quebra
try:
    import rasterio.crs
    print("rasterio.crs importado com sucesso!")
except Exception as e:
    print("\nERRO REAL NO RASTERIO:")
    print(e)

try:
    import localtileserver
    print("localtileserver importado com sucesso!")
except Exception as e:
    print("\nERRO REAL NO LOCALTILESERVER:")
    print(e)

O arquivo proj.db existe? False
Caminho esperado: C:\Python314\Library\share\proj\proj.db
rasterio.crs importado com sucesso!

ERRO REAL NO LOCALTILESERVER:
The EPSG code is unknown. PROJ: proj_create_from_database: C:\Program Files\PostgreSQL\17\share\contrib\postgis-3.5\proj\proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 6 is expected. It comes from another PROJ installation.


Definimos  as  URLs  para  cada  dataset  hospedado  no  Source  Cooperative,  depois  usamos
```geoai.download_file()```  para buscá-los localmente. A função baixa cada arquivo apenas se ele ainda
não existir no diretório de trabalho:

In [19]:
naip_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-naip.tif"
hag_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-hag.tif"
buildings_url = ("https://data.source.coop/opengeos/geoai/las-vegas-buildings-train.geojson")
buildings_mask_url = ("https://data.source.coop/opengeos/geoai/las-vegas-buildings-mask.tif")
naip_path = geoai.download_file(naip_url)
hag_path = geoai.download_file(hag_url)
buildings_path = geoai.download_file(buildings_url)
mask_path = geoai.download_file(buildings_mask_url)

Com os dados baixados, criamos um mapa e adicionamos a imagem NAIP como uma camada raster. O parâmetro  ```layer_name```  define o rótulo que aparece no controle de camadas:

In [20]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Image")
m

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

In [21]:
# para vericar o kernel python que está sendo usado no notebook
import sys
print(sys.executable)

C:\Python314\python.exe


Para imagens multibanda como NAIP,  ```add_raster()```  compõe automaticamente as três primeiras bandas como uma imagem RGB. Para rasters de banda única, você pode especifi car um colormap e intervalo de valores usando  ```vmin```  e  ```vmax```  para controlar como os valores de pixel são mapeados a cores. Aqui adicionamos o raster de altura acima do solo ao mesmo mapa, defi nindo  ```vmin=0```  e  ```vmax=10```  para limitar a exibição a 10 metros para que estruturas na escala de edificações se destaquem claramente contra o solo:

m.add_raster(hag_path, vmin=0, vmax=10, colormap="terrain", layer_name="Height Above Ground" )

Note que  ```add_raster()```  aceita tanto caminhos de arquivos locais quanto URLs remotas. Quando recebe uma URL, transmite os dados diretamente. Você pode escolher qualquer colormap do matplotlib (por exemplo,  ```"terrain"``` ,  ```"viridis"``` ,  ```"RdYlGn"``` ) para corresponder ao tipo de dado. Dados de altura funcionam bem com  "terrain" , enquanto variáveis contínuas como temperatura são adequadas para "coolwarm" . Use o controle de camadas no canto superior direito do mapa para alternar entre a imagem NAIP e a camada de altura acima do solo.

### Cloud-Optimized GeoTIFF (COG)
Cloud-Optimized GeoTIFFs permitem transmitir dados raster diretamente de servidores remotos sem baixar o arquivo inteiro. O método  ```add_cog_layer()```  do  leafmap  lida com a visualização de COG eficientemente.

In [22]:
m = leafmap.Map()
m.add_cog_layer(naip_url, name="Las Vegas NAIP")
m

Map(center=[-115.28050830697481, 36.29026032766879], controls=(ZoomControl(options=['position', 'zoom_in_text'…

COGs são particularmente úteis ao trabalhar com grandes arquivos de imagens de satélite hospedados em plataformas na nuvem como o Planetary Computer ou AWS Open Data.
### Combinações de Bandas
Imagens multiespectrais se tornam mais informativas quando você combina bandas em visualizações compostas. Composições em cor verdadeira usam as bandas vermelho, verde e azul para produzir imagens que  parecem  fotografias  naturais.  Composições  em  falsa  cor  substituem  bandas  infravermelhas  para
destacar feições invisíveis ao olho humano, como vigor da vegetação ou umidade do solo. Escolher a combinação de bandas certa é uma habilidade fundamental em sensoriamento remoto que impacta diretamente a eficácia com que você pode inspecionar dados de treinamento, identificar feições de interesse e avaliar saídas de modelo.


O parâmetro  indexes  em  add_raster()  permite selecionar quais bandas exibir e em qual ordem. Imagens NAIP contêm quatro bandas: Vermelho (1), Verde (2), Azul (3) e Infravermelho Próximo (4). Ao colocar a banda NIR no canal vermelho, a vegetação aparece em vermelho brilhante, facilitando a distinção de áreas vegetadas de superfícies impermeáveis como estradas e telhados

In [23]:
m = leafmap.Map()
m.add_raster(naip_path, indexes=[4, 1, 2], layer_name="False Color")
m

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

Combinações comuns de bandas NAIP incluem:
* Cor verdadeira (bandas 1, 2, 3): Aparência natural
* Falsa cor (bandas 4, 1, 2): A vegetação aparece em vermelho brilhante, útil para distinguir edificações da vegetação

### Visualizando Dados do Planetary Computer
O Microsoft  Planetary Computer hospeda petabytes de dados geoespaciais, incluindo imagens de satélite, mapas de cobertura do solo, registros climáticos e mais, todos acessíveis através de uma API SpatioTemporal Asset Catalog (STAC). Em vez de baixar grandes datasets antes de poder inspecioná-los, você pode
pesquisar o catálogo, visualizar itens interativamente em um mapa e baixar apenas o que precisa. A biblioteca  ```geoai```  fornece funções de conveniência que encapsulam a API STAC do Planetary Computer, tornando simples pesquisar, visualizar e baixar dados sem sair do Python.

### Navegando por Coleções Disponíveis
A função  ```pc_collection_list()```  recupera todas as coleções disponíveis no Planetary Computer e as retorna como um DataFrame. Isso é útil para descobrir quais datasets existem antes de escrever uma consulta de busca.

In [24]:
collections = geoai.pc_collection_list()
print(f"Total collections: {len(collections)}")
collections.head(10)

Total collections: 138


,id,title,description
0,3dep-lidar-classification,USGS 3DEP Lidar Classification,This collection is derived from the [USGS 3DEP...
1,3dep-lidar-copc,USGS 3DEP Lidar Point Cloud,This collection contains source data from the ...
2,3dep-lidar-dsm,USGS 3DEP Lidar Digital Surface Model,This collection is derived from the [USGS 3DEP...
3,3dep-lidar-dtm,USGS 3DEP Lidar Digital Terrain Model,This collection is derived from the [USGS 3DEP...
4,3dep-lidar-dtm-native,USGS 3DEP Lidar Digital Terrain Model (Native),This collection is derived from the [USGS 3DEP...
5,3dep-lidar-hag,USGS 3DEP Lidar Height above Ground,This COG type is generated using the Z dimensi...
6,3dep-lidar-intensity,USGS 3DEP Lidar Intensity,This collection is derived from the [USGS 3DEP...
7,3dep-lidar-pointsourceid,USGS 3DEP Lidar Point Source,This collection is derived from the [USGS 3DEP...
8,3dep-lidar-returns,USGS 3DEP Lidar Returns,This collection is derived from the [USGS 3DEP...
9,3dep-seamless,USGS 3DEP Seamless DEMs,U.S.-wide digital elevation data at horizontal...


### Pesquisando Itens STAC
Uma vez que você sabe qual coleção deseja, use  ```pc_stac_search()```  para encontrar itens que correspondam aos seus critérios espaciais e temporais. A função aceita um ID de coleção, uma caixa delimitadora no formato  ```[oeste, sul, leste, norte]``` , um intervalo de tempo e fi ltros de consulta opcionais. Aqui
pesquisamos imagens NAIP cobrindo a área de Baltimore:

In [25]:
naip_items = geoai.pc_stac_search(
    collection="naip",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
)
naip_items

[<Item id=md_m_3907643_se_18_1_20130917_20131112>]

Cada item retornado é um objeto STAC Item que contém metadados e links para os ativos de dados reais.
Você pode listar os ativos disponíveis para qualquer item:

In [17]:
geoai.pc_item_asset_list(naip_items[0])

['image', 'metadata', 'thumbnail', 'tilejson', 'rendered_preview']

### Visualizando Imagens NAIP
A  função  view_pc_item()   renderiza  um  item  STAC  em  um  mapa  interativo  transmitindo  tiles  do servidor de tiles do Planetary Computer. Nenhum download é necessário. Isso é especialmente útil para visualizar rapidamente resultados de busca antes de se comprometer com um download.

In [26]:
geoai.view_pc_item(item=naip_items[0])

### Visualizando Dados de Cobertura do Solo
O Planetary Computer também hospeda produtos derivados como classifi cações de cobertura do solo.
Aqui pesquisamos o dataset de cobertura do solo de alta resolução da Baía de Chesapeake e o exibimos com um colormap categórico sobre um mapa base de satélite:

In [27]:
lc_items = geoai.pc_stac_search(
    collection="chesapeake-lc-13",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
    max_items=10,
    )
lc_items
geoai.view_pc_item(item=lc_items[0], colormap_name="tab10", basemap="SATELLITE")

O parâmetro  ```colormap_name```  aceita qualquer nome de colormap do matplotlib, e o parâmetro  basemap adiciona  uma  camada  de  fundo  para  contexto  espacial.  Datasets  categóricos  como  cobertura  do  solo funcionam bem com colormaps qualitativos como  ```"tab10"```  ou  ```"Set3"```  que atribuem cores distintas a cada classe.

### Visualizando Imagens Landsat
Para dados de satélite multiespectrais como Landsat, você pode selecionar bandas específicas ou calcular expressões  de  álgebra  de  bandas  instantaneamente.  O  exemplo  a  seguir  pesquisa  cenas  Landsat  com poucas nuvens do final de 2024 e as visualiza de várias maneiras:

In [28]:
landsat_items = geoai.pc_stac_search(
    collection="landsat-c2-l2",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2024-10-27/2024-12-31",
    query={"eo:cloud_cover": {"lt": 1}},
    max_items=10,
)
landsat_items
geoai.pc_item_asset_list(landsat_items[0])

['qa',
 'ang',
 'red',
 'blue',
 'drad',
 'emis',
 'emsd',
 'trad',
 'urad',
 'atran',
 'cdist',
 'green',
 'nir08',
 'lwir11',
 'swir16',
 'swir22',
 'coastal',
 'mtl.txt',
 'mtl.xml',
 'mtl.json',
 'qa_pixel',
 'qa_radsat',
 'qa_aerosol',
 'tilejson',
 'rendered_preview']

Uma composição em cor verdadeira usando as bandas vermelho, verde e azul:

In [29]:
geoai.view_pc_item(item=landsat_items[0], assets=["red", "green", "blue"])

Uma composição em falsa cor coloca a banda infravermelho próximo no canal vermelho, o que faz a vegetação aparecer em vermelho brilhante:

In [22]:
geoai.view_pc_item(item=landsat_items[0], assets=["nir08", "red", "green"])

Você também pode calcular índices espectrais usando o parâmetro  ```expression``` . Aqui calculamos o Índice de Vegetação por Diferença Normalizada (NDVI) e o exibimos com um colormap verde:

In [23]:
geoai.view_pc_item(
    item=landsat_items[0],
    expression="(nir08-red)/(nir08+red)",
    rescale="-1,1",
    colormap_name="greens",
    name="NDVI",
)

O parâmetro  ```expression```  aceita álgebra de bandas usando nomes de ativos como variáveis, e  ```rescale``` controla o intervalo de valores mapeado ao colormap. Esse cálculo instantâneo é processado no lado do servidor pelo endpoint de renderização de tiles, então nenhum processamento local ou download é
necessário.

### Download de Dados do Planetary Computer
Quando você estiver pronto para trabalhar com os dados localmente, use  ```pc_stac_download()```  para baixar ativos específicos para o disco:

In [30]:
geoai.pc_stac_download(naip_items, output_dir="data", assets=["image","thumbnail"])

{'md_m_3907643_se_18_1_20130917_20131112': {'image': 'data\\md_m_3907643_se_18_1_20130917_20131112_image.tif',
  'thumbnail': 'data\\md_m_3907643_se_18_1_20130917_20131112_thumbnail.jpg'}}

Você também pode ler um ativo diretamente em um xarray DataArray sem salvá-lo em disco, o que é conveniente para análises rápidas:

In [31]:
ds = geoai.read_pc_item_asset(lc_items[0], asset="data")
ds

<xarray.DataArray (band: 1, y: 10000, x: 10000)> Size: 100MB
[100000000 values with dtype=uint8]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 80kB 1.98e+06 1.98e+06 ... 1.97e+06 1.97e+06
  * x            (x) float64 80kB 1.64e+06 1.64e+06 ... 1.65e+06 1.65e+06
    spatial_ref  int64 8B 0
Attributes:
    DataType:            Generic
    AREA_OR_POINT:       Area
    RepresentationType:  THEMATIC
    _FillValue:          0
    scale_factor:        1.0
    add_offset:          0.0

## Trabalhando com Dados Vetoriais em Mapas
Dados  vetoriais  desempenham  um  papel  central  em  GeoAI  como  rótulos  de  treinamento,  limites  de referência e saídas de modelo. Pegadas de edificações, redes viárias, polígonos de uso do solo e caixas
delimitadoras de objetos detectados são todos feições vetoriais que precisam ser visualizadas junto com imagens raster. O  leafmap  fornece métodos flexíveis para adicionar camadas vetoriais de múltiplas
fontes, incluindo arquivos locais, URLs remotas e GeoDataFrames em memória.
    
###  Adicionando GeoJSON e GeoDataFrame
O  leafmap  suporta a adição de dados vetoriais a partir de arquivos GeoJSON, URLs ou GeoDataFrames criados com geopandas. Essa flexibilidade significa que você pode visualizar dados em qualquer estágio do
seu fluxo de trabalho, seja carregando anotações brutas de um arquivo ou exibindo resultados processados de um GeoDataFrame em memória. Um padrão comum em GeoAI é sobrepor anotações vetoriais sobre a
imagem de origem para verificar se os rótulos estão alinhados com feições visíveis.

In [32]:
import geopandas as gpd

gdf = gpd.read_file(buildings_path)
print(f"Features: {len(gdf)}")
gdf.head()

m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Building Footprints", zoom_to_layer=True)
m

Features: 23771


ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

Você  também  pode  adicionar  GeoJSON  diretamente  de  uma  URL  sem  primeiro  carregá-lo  em  um ```GeoDataFrame```:

In [9]:
m = leafmap.Map()
m.add_raster(naip_path)
#m.add_geojson(buildings_url, layer_name="Buildings", zoom_to_layer=True)
m

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

### Estilizando Camadas Vetoriais
Estilização personalizada ajuda a distinguir diferentes tipos de feições e chamar atenção para padrões importantes. Você pode controlar cor de preenchimento, cor do contorno, largura da linha e opacidade

In [10]:
style = {
    "color": "red",
    "weight": 2,
    "fillColor": "yellow",
    "fillOpacity": 0.3,
}
m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Styled Buildings", style=style, zoom_to_layer=True)
m

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

O dicionário  ```style```  segue a convenção de opções de caminho do Leaflet. As propriedades principais incluem  ```color```  (contorno),  ```weight```  (largura do contorno em pixels),  ```fillColor```  e  ```fillOpacity```  (0 para transparente, 1 para totalmente opaco).
### Adicionando Marcadores e Pontos
Para dados de pontos como locais de amostragem ou pontos de interesse, você pode adicionar marcadores ao mapa.

In [33]:
m = leafmap.Map(center=[36.1617, -115.1524], zoom=11)
m.add_marker(location=[36.1617, -115.1524])
m

Map(center=[36.1617, -115.1524], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

Para datasets com muitos pontos, considere usar agrupamento de marcadores para evitar desordem visual em níveis de zoom baixos. O  ```leafmap```  pode agrupar camadas densas de pontos para que permaneçam legíveis em escalas de mapa mais amplas.

##  Mapas de Painel Dividido para Comparação

A comparação está no coração de muitas tarefas de GeoAI. A detecção de mudanças requer comparar imagens de datas diferentes. A avaliação de modelo exige comparar predições com a verdade de campo. A seleção de dados envolve comparar imagens de diferentes sensores ou estações para escolher a melhor fonte. Mapas de painel dividido fornecem uma maneira estruturada de fazer essas comparações sem perder o contexto espacial.
### Comparação Lado a Lado
Mapas de painel dividido são inestimáveis para comparar dois datasets, como imagens de datas diferentes ou predições de modelo versus verdade de campo. O método  ```split_map()```  do  ```leafmap```  cria um mapa com um divisor arrastável entre duas camadas. Essa abordagem é particularmente eficaz porque ambos os datasets compartilham a mesma extensão geográfica e nível de zoom, garantindo que a comparação seja espacialmente consistente.


O método  ```split_map()```  aceita fontes de camada comuns como arquivos raster e arquivos vetoriais, e ```left_args```  e  ```right_args```  permitem passar opções de renderização para cada lado. Aqui comparamos a imagem NAIP, exibida como uma composição em falsa cor, com o raster de altura acima do solo:

In [34]:
m = leafmap.Map()
m.split_map(
    left_layer=naip_path,
    right_layer=hag_path,
    left_args={"indexes": [4, 1, 2]},
    right_args={"vmin": 0, "vmax": 10, "cmap": "terrain"},
    left_label="NAIP",
    right_label="HAG",
)
m

The provided layers are invalid!


ValueError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

Arraste o controle deslizante para a esquerda e para a direita para revelar cada camada. Note como as áreas com estruturas altas na imagem NAIP correspondem a valores altos no raster HAG. Esse tipo de comparação é particularmente útil para verifi car se produtos derivados como modelos de altura estão
alinhados com a imagem de origem.

## Visualizando Resultados de Modelo
A visualização se torna ainda mais crítica quando você começa a produzir saídas de modelo. Métricas de acurácia brutas dizem como um modelo funciona em média, mas a inspeção visual revela onde e por que ele falha. Um modelo pode alcançar 90% de acurácia geral enquanto ignora completamente edificações em sombras, confunde estacionamentos com telhados ou produz limites irregulares ao longo de bordas de água. Esses padrões espaciais de erro são invisíveis em estatísticas resumidas, mas imediatamente aparentes em um mapa.

### Sobrepondo Predições
Após  executar  um  modelo  de  GeoAI,  você  tipicamente  tem  saídas  de  predição  como  arquivos  raster (máscaras  de  segmentação)  ou  arquivos  vetoriais  (objetos  detectados).  Sobrepor  estes  na  imagem  de origem ajuda a avaliar o desempenho do modelo visualmente. A técnica chave é exibir a sobreposição
com transparência parcial para que a imagem subjacente permaneça visível, permitindo avaliar se as feições destacadas correspondem a objetos reais no terreno. Para demonstrar o fluxo de trabalho, usamos a máscara rasterizada de edificações da mesma forma que você exibiria um raster de predição

In [35]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Imagery")

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

Definir ```opacity```  para um valor menor que 1 permite que a imagem subjacente apareça, facilitando ver onde a máscara se alinha com estruturas visíveis. O parâmetro  ```nodata=0```  garante que pixels não-edificação (valor 0) sejam renderizados como transparentes, de modo que apenas as áreas de edificação
sejam destacadas sobre a imagem NAIP.
###  Comparando Rótulos com Imagem de Origem
Antes de comparar predições com rótulos, muitas vezes é útil verificar se os rótulos de referência estão alinhados com a imagem de origem. Mapas de painel dividido funcionam bem para essa etapa de controle de qualidade, permitindo arrastar um controle deslizante para comparar anotações vetoriais com a imagem
subjacente. Aqui usamos  ```split_map()```  para comparar pegadas de edificações, estilizadas com contornos vermelhos e preenchimento semitransparente, com a imagem NAIP:

Arraste o controle deslizante para comparar as anotações de pegadas de edificações com a imagem NAIP subjacente. Isso revela se as anotações traçam com precisão os limites das edificações, se alguma edificação foi perdida e se estruturas não-edificação foram rotuladas por engano. Uma vez que os rótulos pareçam
corretos, você pode substituir uma camada de predição no mesmo fl uxo de trabalho de comparação.

## Melhores Práticas para Visualização em GeoAI
Visualização eficaz requer mais do que simplesmente plotar dados em um mapa. As seguintes melhores práticas ajudarão você a criar visualizações informativas e profissionais.


**Escolha colormaps apropriados para seu tipo de dado.** Colormaps sequenciais como  ```"viridis"```  ou ```"terrain"```  funcionam bem para dados contínuos como elevação ou NDVI. Colormaps divergentes como ```"RdBu"```  são melhores para dados com um ponto central signifi cativo, como anomalias de temperatura ou  valores  de  mudança.  Colormaps  categóricos  devem  usar  cores  distintas  e  facilmente  distinguíveis para cada classe. Evite colormaps arco-íris, que podem ser enganosos porque carecem de uma ordenação perceptual natural.


**Sempre inclua contexto espacial.** Um mapa de predição sem mapa base ou feições de referência é difícil de interpretar. Inclua imagens de satélite, limites administrativos ou outras camadas de referência que ajudem os visualizadores a se orientar e entender o cenário geográfico dos seus resultados.
Use estilização consistente entre comparações. Ao comparar saídas de modelo de diferentes experimentos ou períodos de tempo, use o mesmo colormap, intervalo de valores e configurações de opacidade para todas as camadas. Estilização inconsistente pode criar diferenças visuais falsas que não refletem
diferenças reais nos dados.


**Rotule suas camadas claramente.** Dê a cada camada um nome descritivo que apareça no controle de camadas. Nomes como “Building Predictions (U-Net)” são muito mais úteis do que “Layer 1” quando você tem múltiplas camadas ativas simultaneamente.


**Considere seu público.** Colegas técnicos podem apreciar mapas multicamada detalhados com interatividade completa, enquanto tomadores de decisão podem precisar de visualizações mais simples com legendas e anotações claras. Adapte a complexidade e o formato das suas visualizações às pessoas que as utilizarão.

In [36]:
import leafmap
m = leafmap.Map()
m.split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
)
m

The provided layers are invalid!


ValueError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

A biblioteca  geoai  também fornece uma função de conveniência  ```create_split_map()```  que adiciona uma camada de mapa base abaixo de ambos os painéis, fornecendo contexto espacial adicional:

In [ ]:
geoai.create_split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=naip_path,
)

## Exercícios
## Exercício 1: Criar um Mapa com Mapa Base Personalizado 
Crie um mapa interativo centralizado em uma cidade de sua escolha. Adicione pelo menos três mapas base diferentes e defina um nível de zoom apropriado. Experimente o controle de camadas para alternar entre os mapas base.
### Exercício 2: Visualizar um Raster com Múltiplos Colormaps
Carregue o raster de Altura Acima do Solo de Las Vegas. Adicione-o a um mapa usando três colormaps diferentes (por exemplo,  ```"terrain"``` ,  ```"viridis"```  e  ```"gray"``` ), cada um como uma camada separada. Use o controle de camadas para comparar como cada colormap representa os valores de altura. Qual colormap comunica diferenças de altura de forma mais eficaz?
###  Exercício 3: Estilizar e Explorar Dados Vetoriais
Carregue o GeoJSON de pegadas de edifi cações de Las Vegas 62 . Exiba as edificações em um mapa base de satélite com um estilo personalizado que use uma cor de contorno brilhante e preenchimento semitransparente. Adicione um marcador no centroide da maior edificação com um popup mostrando sua área.
###  Exercício 4: Construir uma Comparação em Painel Dividido
Crie um mapa de painel dividido usando  ```split_map()```  que compare a imagem NAIP  com o raster de altura acima do solo para a área de estudo de Las Vegas. Experimente diferentes combinações de bandas para o lado NAIP (cor verdadeira vs. falsa cor) e diferentes colormaps para o lado HAG. Qual combinação
facilita mais a identificação de edificações?
###  Exercício 5: Sobrepor e Comparar Camadas de Edificações
Usando os datasets de Las Vegas, crie um mapa que exiba todas as quatro camadas: a imagem NAIP como base, a máscara de edificações com opacidade parcial, o raster de altura acima do solo e as pegadas de edificações como contornos. Alterne camadas para identificar pelo menos duas áreas onde a máscara de
edificações e as pegadas de edificações diferem.